In [ ]:
# create_file
import os
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.loader import DataLoader
from Bio.PDB import PDBParser, PDBIO
sys.path.append("/home/lizihao/Work/enzyme_prediction/src/simple2/scripts")

# 导入模型类
from GNN_model import PocketGNNWithAttention

# 全局变量
DATASET_PATH = "/home/lizihao/Work/enzyme_prediction/src/simple2/data/processed/dataset_NAN_nopqr_rbf.pt"
MODEL_PATH = "/home/lizihao/Work/enzyme_prediction/src/simple2/outputs/nopqr_attention_rbf/best_model.pt"
SAVE_DIR = "/home/lizihao/Work/enzyme_prediction/src/simple2/analysis_results/pocket_visualization"
POCKET_DIR = "/home/lizihao/Work/enzyme_prediction/src/simple2/data/processed/pockets"
DEVICE = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

def compute_input_gradient(model, data, target_index=0):
    """
    计算输入特征对模型输出的贡献度
    
    参数:
        model: 训练好的GNN模型
        data: PyG数据对象
        target_index: 目标输出索引(0=kcat, 1=Km)
    """
    model.eval()
    data.x.requires_grad_(True)
    output = model(data)
    score = output[:, target_index].sum()
    grad = torch.autograd.grad(score, data.x, retain_graph=True)[0]
    attribution = (data.x * grad).sum(dim=1)
    return attribution.detach().cpu().numpy()

def visualize_node_attribution(attribution, save_path='atom_importance_bar.png', topk=20):
    """
    可视化原子贡献度的条形图
    
    参数:
        attribution: 每个原子的贡献度数组
        save_path: 图像保存路径
        topk: 显示前k个最重要的原子
    """
    topk_idx = np.argsort(np.abs(attribution))[-topk:][::-1]
    topk_vals = attribution[topk_idx]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=np.arange(topk), y=topk_vals)
    plt.title("Top-k Atom Contribution (Input x Gradient)")
    plt.xlabel("Atom Index")
    plt.ylabel("Importance Score")
    plt.savefig(save_path)
    plt.close()
    print(f"✅ 保存了贡献度条形图：{save_path}")

def visualize_attribution_pymol(data, attribution, output_pdb="atom_importance.pdb", normalize=True):
    """
    将归因分数写入PDB文件的B因子字段，用于PyMOL可视化
    
    参数:
        data: PyG的Data对象，包含原始PDB信息
        attribution: 计算得到的原子归因分数
        output_pdb: 输出的PDB文件路径
        normalize: 是否将贡献度标准化到[0,100]范围
    """
    from Bio.PDB import PDBParser, PDBIO
    import numpy as np
    import os
    
    # 加载原始PDB文件
    pdb_path = os.path.join(POCKET_DIR, data.pdb_id)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pocket', pdb_path)
    
    # 标准化贡献值（便于PyMOL着色）
    if normalize:
        min_val = attribution.min()
        max_val = attribution.max()
        if max_val > min_val:
            norm_attr = (attribution - min_val) * 100 / (max_val - min_val)
        else:
            norm_attr = np.zeros_like(attribution)
    else:
        norm_attr = attribution
        
    # 将贡献值写入B因子
    atom_idx = 0
    for atom in structure.get_atoms():
        if atom.element != 'H':  # 跳过氢原子以保持索引一致性
            atom.set_bfactor(float(norm_attr[atom_idx]))
            atom_idx += 1
    
    # 保存修改后的PDB文件
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_pdb)
    
    print(f"✅ 保存了带有贡献度的PDB文件：{output_pdb}")
    print(f"🔍 在PyMOL中查看方法:\n"
          f"  1. pymol {output_pdb}\n"
          f"  2. 执行以下PyMOL命令:\n"
          f"     spectrum b, blue_white_red\n"
          f"     show sticks\n"
          f"     label (name CA), \"%%resi %%resn\"\n"
          f"     set label_size, 1.0")

def visualize_in_pymol(data, attribution, target_name="kcat", temp_pdb="temp_attribution.pdb"):
    """
    直接在PyMOL中启动可视化
    
    参数:
        data: PyG数据对象
        attribution: 原子贡献度数组
        target_name: 目标名称(kcat或Km)
        temp_pdb: 临时PDB文件路径
    """
    from Bio.PDB import PDBParser, PDBIO
    import os
    import subprocess
    import numpy as np
    
    # 首先生成带有贡献度的PDB
    pdb_path = os.path.join(POCKET_DIR, data.pdb_id)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pocket', pdb_path)
    
    # 标准化贡献值
    min_val = attribution.min()
    max_val = attribution.max()
    if max_val > min_val:
        norm_attr = (attribution - min_val) * 100 / (max_val - min_val)
    else:
        norm_attr = np.zeros_like(attribution)
        
    # 将贡献值写入B因子
    atom_idx = 0
    for atom in structure.get_atoms():
        if atom.element != 'H':
            atom.set_bfactor(float(norm_attr[atom_idx]))
            atom_idx += 1
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(temp_pdb)
    
    # 创建PyMOL脚本
    pymol_script = f"""
import pymol
pymol.cmd.load("{temp_pdb}", "pocket")
pymol.cmd.spectrum("b", "blue_white_red", "pocket")
pymol.cmd.show("sticks")
pymol.cmd.set("stick_radius", "0.2")
pymol.cmd.select("ligand", "resn UNL or chain ''")
pymol.cmd.color("yellow", "ligand")
pymol.cmd.show("spheres", "ligand")
pymol.cmd.set("sphere_scale", "0.4", "ligand")
pymol.cmd.label("name CA", "%resi %resn")
pymol.cmd.set("label_size", "1.0")
pymol.cmd.bg_color("white")
pymol.cmd.set("ray_opaque_background", "off")
pymol.cmd.orient()
pymol.cmd.set("depth_cue", "0")
pymol.cmd.set("ray_shadows", "0")
pymol.cmd.png("{target_name}_contribution.png", width=1200, height=1000, dpi=300, ray=1)
    """
    
    with open("vis_script.py", "w") as f:
        f.write(pymol_script)
    
    # 运行PyMOL
    print("🚀 启动PyMOL可视化中...")
    subprocess.run(["pymol", "-c", "vis_script.py"])
    print(f"✅ 已生成高质量渲染图: {target_name}_contribution.png")

def main():
    """主函数：执行完整链路调用"""
    # 创建输出目录
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    # 加载数据集
    print("📂 加载数据集...")
    dataset = torch.load(DATASET_PATH)
    
    # 加载模型
    print("🧠 加载模型...")
    model = PocketGNNWithAttention(
        node_input_dim=dataset[0].x.shape[1],
        edge_input_dim=dataset[0].edge_attr.shape[1],
    )
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    
    # 选择样本进行分析
    sample_idx = 0  # 第一个样本
    data = dataset[sample_idx].to(DEVICE)
    
    # kcat的原子贡献度分析
    print("📊 计算kcat原子贡献度...")
    kcat_attribution = compute_input_gradient(model, data, target_index=0)
    
    # 条形图可视化
    visualize_node_attribution(
        kcat_attribution, 
        save_path=os.path.join(SAVE_DIR, "kcat_atom_importance.png"), 
        topk=20
    )
    
    # PyMOL可视化（静态PDB文件）
    visualize_attribution_pymol(
        data, 
        kcat_attribution, 
        output_pdb=os.path.join(SAVE_DIR, "kcat_atom_importance.pdb")
    )
    
    # 直接生成高质量渲染图
    # visualize_in_pymol(
    #     data, 
    #     kcat_attribution, 
    #     target_name="kcat",
    #     temp_pdb=os.path.join(SAVE_DIR, "temp_kcat.pdb")
    # )
    
    # Km的原子贡献度分析
    print("📊 计算Km原子贡献度...")
    km_attribution = compute_input_gradient(model, data, target_index=1)
    
    # 条形图可视化
    visualize_node_attribution(
        km_attribution, 
        save_path=os.path.join(SAVE_DIR, "km_atom_importance.png"), 
        topk=20
    )
    
    # PyMOL可视化（静态PDB文件）
    visualize_attribution_pymol(
        data, 
        km_attribution, 
        output_pdb=os.path.join(SAVE_DIR, "km_atom_importance.pdb")
    )
    
    # 直接生成高质量渲染图
    visualize_in_pymol(
        data, 
        km_attribution, 
        target_name="km",
        temp_pdb=os.path.join(SAVE_DIR, "temp_km.pdb")
    )
    
    print(f"✅ 分析完成！结果已保存到: {SAVE_DIR}")
    print("📝 总结:")
    print("  1. 条形图显示了贡献最大的原子 (.png)")
    print("  2. PDB文件中的B因子包含了归因分数 (.pdb)")
    print("  3. 高质量渲染图直观展示了结构 (kcat_contribution.png, km_contribution.png)")

if __name__ == "__main__":
    main()

📂 加载数据集...


/tmp/ipykernel_3558446/345134616.py:185: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load(DATASET_PATH)


🧠 加载模型...


/tmp/ipykernel_3558446/345134616.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))


RuntimeError: Error(s) in loading state_dict for PocketGNNWithAttention:
	Unexpected key(s) in state_dict: "att_layers.3.att_src", "att_layers.3.att_dst", "att_layers.3.bias", "att_layers.3.lin.weight", "att_layers.4.att_src", "att_layers.4.att_dst", "att_layers.4.bias", "att_layers.4.lin.weight", "att_layers.5.att_src", "att_layers.5.att_dst", "att_layers.5.bias", "att_layers.5.lin.weight". 
	size mismatch for node_encoder.weight: copying a param with shape torch.Size([256, 52]) from checkpoint, the shape in current model is torch.Size([128, 52]).
	size mismatch for node_encoder.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for att_layers.0.att_src: copying a param with shape torch.Size([1, 8, 32]) from checkpoint, the shape in current model is torch.Size([1, 4, 32]).
	size mismatch for att_layers.0.att_dst: copying a param with shape torch.Size([1, 8, 32]) from checkpoint, the shape in current model